In [ ]:
# -*- coding: utf-8 -*-
"""
Comparação de compensação térmica:
PARK vs AUTOENCODER RESIDUAL

Métricas:
- RMSD
- CCDM

Gráficos:
1) RMSD e CCDM por temperatura para cada classe de dano
2) RMSD e CCDM médios por classe de dano
3) Curva exemplo: referência, original, Park e Autoencoder
4) Boxplot por método e dano
5) Tabela de monotonicidade D0 < D1 < D2

Autor: Luiz Eduardo Abdala José
"""

# ============================================================
# 1) BIBLIOTECAS
# ============================================================

import os
import re
import time
import copy
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=UserWarning)


# ============================================================
# 2) PARÂMETROS GERAIS
# ============================================================

ARQ_BASE = "base-completo--.pkl"

REF_TEMP = 30

FREQ_MIN_KHZ = 40
FREQ_MAX_KHZ = 50

SMOOTH_WIN = 5

OUTPUT_DIR = f"Comparacao_Park_AE_RMSD_CCDM_REF{REF_TEMP}C_{FREQ_MIN_KHZ}-{FREQ_MAX_KHZ}kHz"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 3) PARÂMETROS DO PARK
# ============================================================

PARK_MAX_SHIFT_FRAC = 0.25
PARK_NSTEPS = 201
PARK_SMOOTH_WIN = 5


# ============================================================
# 4) PARÂMETROS DO AUTOENCODER
# ============================================================

AE_EPOCHS = 800
AE_BATCH_SIZE = 16
AE_LR = 1e-3
AE_LATENT_DIM = 64

AE_VAL_FRAC = 0.20
AE_PATIENCE = 120

ALPHA_COMP_AE = 0.85


# ============================================================
# 5) SEMENTES
# ============================================================

np.random.seed(42)
torch.manual_seed(42)


# ============================================================
# 6) FUNÇÕES DE FREQUÊNCIA E PRÉ-PROCESSAMENTO
# ============================================================

def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None


def get_freq_columns(df, fmin_khz, fmax_khz):
    cols = []
    freqs = []

    for c in df.columns:
        f = extract_freq_hz(c)

        if f is not None:
            f_khz = f / 1e3

            if fmin_khz <= f_khz <= fmax_khz:
                cols.append(c)
                freqs.append(f)

    order = np.argsort(freqs)

    fcols = [cols[i] for i in order]
    fhz = np.array(freqs, dtype=float)[order]

    return fcols, fhz


def moving_average(arr, win):
    if win <= 1 or win % 2 == 0:
        return arr.copy()

    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode="edge")

    kernel = np.ones(win) / win
    smooth = np.convolve(arr_pad, kernel, mode="valid")

    return smooth[:len(arr)]


def add_extra_features_matrix(X):
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    amp = (X.max(axis=1) - X.min(axis=1)).reshape(-1, 1)

    return np.hstack([X, mu, sd, amp])


def get_reference_curve(df, fcols, ref_temp):
    df_sem = df[df["falha"] == 0].copy()

    pool = df_sem.loc[
        np.isclose(df_sem["temperatura_c"], ref_temp),
        fcols
    ].to_numpy(float)

    if len(pool) == 0:
        raise ValueError(
            f"Nenhuma curva saudável encontrada em REF_TEMP = {ref_temp} °C."
        )

    y_ref = np.median(pool, axis=0)

    return y_ref


# ============================================================
# 7) MÉTRICAS RMSD E CCDM
# ============================================================

def rmsd(y, ref):
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)

    return float(np.sqrt(np.mean((y - ref) ** 2)))


def ccdm(y, ref):
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)

    y0 = y - np.mean(y)
    r0 = ref - np.mean(ref)

    num = float(np.sum(y0 * r0))
    den = float(np.sqrt(np.sum(y0 ** 2) * np.sum(r0 ** 2)) + 1e-18)

    corr = num / den

    return float(1 - corr)


def calcular_metricas(df_curvas, fcols, y_ref, metodo):
    X = df_curvas[fcols].to_numpy(float)

    df_out = df_curvas.copy()

    df_out["RMSD"] = [rmsd(x, y_ref) for x in X]
    df_out["CCDM"] = [ccdm(x, y_ref) for x in X]
    df_out["Metodo"] = metodo

    return df_out


# ============================================================
# 8) MÉTODO PARK
# ============================================================

def shift_interp(x, fHz, tau):
    f_shift = fHz + tau

    return np.interp(
        fHz,
        f_shift,
        x,
        left=x[0],
        right=x[-1]
    )


def park_single(x, ref, fHz):
    df_band = fHz[-1] - fHz[0]
    tau_max = PARK_MAX_SHIFT_FRAC * df_band

    best_err = np.inf
    best_tau = 0.0
    best_dS = 0.0

    taus = np.linspace(-tau_max, tau_max, PARK_NSTEPS)

    for tau in taus:
        x_shift = shift_interp(x, fHz, tau)

        dS = np.mean(ref - x_shift)

        y_try = x_shift + dS

        err = np.sum((ref - y_try) ** 2)

        if err < best_err:
            best_err = err
            best_tau = tau
            best_dS = dS

    y_comp = shift_interp(x, fHz, best_tau) + best_dS
    y_comp = moving_average(y_comp, PARK_SMOOTH_WIN)

    return y_comp


def compensar_park(df, fcols, fHz, y_ref):
    print("\n====================================================")
    print("APLICANDO PARK")
    print("====================================================")

    X_all = df[fcols].to_numpy(float)

    Y_comp = np.zeros_like(X_all)

    for i in range(len(X_all)):
        Y_comp[i] = park_single(
            x=X_all[i],
            ref=y_ref,
            fHz=fHz
        )

        if (i + 1) % 20 == 0 or (i + 1) == len(X_all):
            print(f"Park: {i+1}/{len(X_all)} curvas compensadas")

    df_comp = df.copy()
    df_comp[fcols] = Y_comp

    return df_comp


# ============================================================
# 9) AUTOENCODER RESIDUAL
# ============================================================

class ResidualAutoencoder(nn.Module):
    def __init__(self, n_in, n_out, latent_dim=64):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(n_in, 512),
            nn.ReLU(),

            nn.Linear(512, 256),
            nn.ReLU(),

            nn.Linear(256, latent_dim),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),

            nn.Linear(256, 512),
            nn.ReLU(),

            nn.Linear(512, n_out)
        )

    def forward(self, x):
        z = self.encoder(x)
        delta = self.decoder(z)

        return delta


def train_autoencoder_residual(
    X_input,
    Y_target,
    epochs=800,
    batch_size=16,
    lr=1e-3,
    latent_dim=64,
    val_frac=0.20,
    patience=120
):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("\n====================================================")
    print("TREINANDO AUTOENCODER RESIDUAL")
    print("====================================================")
    print(f"Dispositivo usado no Autoencoder: {device}")

    sx = StandardScaler()
    sy = StandardScaler()

    Xs = sx.fit_transform(X_input)
    Ys = sy.fit_transform(Y_target)

    N = Xs.shape[0]

    idx = np.arange(N)
    np.random.shuffle(idx)

    n_val = int(np.floor(val_frac * N))

    val_idx = idx[:n_val]
    train_idx = idx[n_val:]

    if len(val_idx) == 0:
        val_idx = train_idx.copy()

    X_train = torch.tensor(Xs[train_idx], dtype=torch.float32)
    Y_train = torch.tensor(Ys[train_idx], dtype=torch.float32)

    X_val = torch.tensor(Xs[val_idx], dtype=torch.float32)
    Y_val = torch.tensor(Ys[val_idx], dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(X_train, Y_train),
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        TensorDataset(X_val, Y_val),
        batch_size=batch_size,
        shuffle=False
    )

    model = ResidualAutoencoder(
        n_in=X_input.shape[1],
        n_out=Y_target.shape[1],
        latent_dim=latent_dim
    ).to(device)

    opt = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=1e-5
    )

    loss_fn = nn.MSELoss()

    best_val_loss = np.inf
    best_state = copy.deepcopy(model.state_dict())
    epochs_sem_melhora = 0

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": []
    }

    for ep in range(1, epochs + 1):
        model.train()

        train_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            opt.zero_grad()

            pred = model(xb)
            loss = loss_fn(pred, yb)

            loss.backward()
            opt.step()

            train_loss += loss.item()

        train_loss /= max(1, len(train_loader))

        model.eval()

        val_loss = 0.0

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                pred = model(xb)
                loss = loss_fn(pred, yb)

                val_loss += loss.item()

        val_loss /= max(1, len(val_loader))

        history["epoch"].append(ep)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val_loss - 1e-7:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_sem_melhora = 0
        else:
            epochs_sem_melhora += 1

        if ep == 1 or ep % 50 == 0:
            print(
                f"Epoch {ep:4d}/{epochs} | "
                f"train_loss = {train_loss:.6f} | "
                f"val_loss = {val_loss:.6f}"
            )

        if epochs_sem_melhora >= patience:
            print(
                f"Early stopping na epoch {ep}. "
                f"Melhor val_loss = {best_val_loss:.6f}"
            )
            break

    model.load_state_dict(best_state)

    return model, sx, sy, device, pd.DataFrame(history)


def predict_autoencoder_residual(model, sx, sy, device, X_input):
    model.eval()

    Xs = sx.transform(X_input)

    X_tensor = torch.tensor(Xs, dtype=torch.float32).to(device)

    with torch.no_grad():
        pred_scaled = model(X_tensor).cpu().numpy()

    Delta_hat = sy.inverse_transform(pred_scaled)

    return Delta_hat


def compensar_autoencoder(df, fcols, fHz, y_ref):
    df_sem = df[df["falha"] == 0].copy()

    X_sem = df_sem[fcols].to_numpy(float)
    T_sem = df_sem["temperatura_c"].to_numpy(float)

    Y_target = y_ref[None, :] - X_sem

    X_aug_sem = add_extra_features_matrix(X_sem)

    X_in_sem = np.hstack([
        X_aug_sem,
        T_sem.reshape(-1, 1)
    ])

    ae_model, sx, sy, device, history = train_autoencoder_residual(
        X_input=X_in_sem,
        Y_target=Y_target,
        epochs=AE_EPOCHS,
        batch_size=AE_BATCH_SIZE,
        lr=AE_LR,
        latent_dim=AE_LATENT_DIM,
        val_frac=AE_VAL_FRAC,
        patience=AE_PATIENCE
    )

    print("\n====================================================")
    print("APLICANDO AUTOENCODER RESIDUAL")
    print("====================================================")

    X_all = df[fcols].to_numpy(float)
    T_all = df["temperatura_c"].to_numpy(float)

    X_aug_all = add_extra_features_matrix(X_all)

    X_in_all = np.hstack([
        X_aug_all,
        T_all.reshape(-1, 1)
    ])

    Delta_hat = predict_autoencoder_residual(
        model=ae_model,
        sx=sx,
        sy=sy,
        device=device,
        X_input=X_in_all
    )

    Y_comp = X_all + ALPHA_COMP_AE * Delta_hat

    for i in range(len(Y_comp)):
        Y_comp[i] = moving_average(Y_comp[i], SMOOTH_WIN)

    df_comp = df.copy()
    df_comp[fcols] = Y_comp

    return df_comp, ae_model, history


# ============================================================
# 10) RESUMOS NUMÉRICOS
# ============================================================

def resumo_geral(df_long, metodos=("Park", "Autoencoder")):
    df_use = df_long[df_long["Metodo"].isin(metodos)].copy()

    tabela = (
        df_use
        .groupby(["Metodo", "falha"])[["RMSD", "CCDM"]]
        .agg(["mean", "std", "min", "max"])
        .round(6)
    )

    return tabela


def resumo_por_temperatura(df_long, metodos=("Park", "Autoencoder")):
    df_use = df_long[df_long["Metodo"].isin(metodos)].copy()

    tabela = (
        df_use
        .groupby(["Metodo", "temperatura_c", "falha"])[["RMSD", "CCDM"]]
        .mean()
        .reset_index()
        .sort_values(["Metodo", "temperatura_c", "falha"])
    )

    return tabela


def checar_monotonicidade(df_long, metodos=("Park", "Autoencoder")):
    df_use = df_long[df_long["Metodo"].isin(metodos)].copy()

    registros = []

    for metodo in sorted(df_use["Metodo"].unique()):
        df_m = df_use[df_use["Metodo"] == metodo]

        temps = sorted(df_m["temperatura_c"].unique())

        for T in temps:
            df_t = df_m[np.isclose(df_m["temperatura_c"], T)]

            danos_presentes = set(df_t["falha"].unique())

            if not {0, 1, 2}.issubset(danos_presentes):
                continue

            for metrica in ["RMSD", "CCDM"]:
                medias = {}

                for d in [0, 1, 2]:
                    medias[d] = df_t.loc[
                        df_t["falha"] == d,
                        metrica
                    ].mean()

                ok = medias[0] < medias[1] < medias[2]

                registros.append({
                    "Metodo": metodo,
                    "Temperatura": T,
                    "Metrica": metrica,
                    "D0": medias[0],
                    "D1": medias[1],
                    "D2": medias[2],
                    "Monotonico_D0_D1_D2": ok
                })

    df_mono = pd.DataFrame(registros)

    if len(df_mono) == 0:
        resumo = pd.DataFrame()
        return df_mono, resumo

    resumo = (
        df_mono
        .groupby(["Metodo", "Metrica"])["Monotonico_D0_D1_D2"]
        .mean()
        .mul(100)
        .reset_index()
        .rename(columns={
            "Monotonico_D0_D1_D2": "Percentual_monotonico_%"
        })
    )

    return df_mono, resumo


# ============================================================
# 11) ESTILO DOS GRÁFICOS
# ============================================================

def aplicar_estilo_artigo():
    plt.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": 18,
        "axes.labelsize": 20,
        "axes.titlesize": 20,
        "xtick.labelsize": 17,
        "ytick.labelsize": 17,
        "legend.fontsize": 15,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "pdf.fonttype": 42,
        "ps.fonttype": 42
    })


# ============================================================
# 12) GRÁFICO 1 — RMSD E CCDM POR TEMPERATURA
# ============================================================

def plot_metricas_por_temperatura(
    df_long,
    dano=0,
    metodos=("Park", "Autoencoder"),
    salvar=True
):
    aplicar_estilo_artigo()

    df_use = df_long[
        (df_long["falha"] == dano) &
        (df_long["Metodo"].isin(metodos))
    ].copy()

    fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), dpi=300)

    metricas = ["RMSD", "CCDM"]

    for ax, metrica in zip(axes, metricas):

        for metodo in metodos:
            df_m = df_use[df_use["Metodo"] == metodo]

            g = (
                df_m
                .groupby("temperatura_c")[metrica]
                .mean()
                .reset_index()
                .sort_values("temperatura_c")
            )

            ax.plot(
                g["temperatura_c"],
                g[metrica],
                marker="o",
                linewidth=2,
                label=metodo
            )

        ax.set_xlabel("Temperatura (°C)")
        ax.set_ylabel(metrica)

        ax.grid(False)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_title(f"(a) RMSD — Dano {dano}")
    axes[1].set_title(f"(b) CCDM — Dano {dano}")

    axes[0].legend(
        frameon=True,
        facecolor="white",
        edgecolor="none"
    )

    plt.tight_layout()

    if salvar:
        nome = f"Park_vs_AE_Metricas_por_temperatura_Dano{dano}.png"
        caminho = os.path.join(OUTPUT_DIR, nome)

        plt.savefig(
            caminho,
            bbox_inches="tight",
            facecolor="white"
        )

        print(f"Figura salva em: {caminho}")

    plt.show()


# ============================================================
# 13) GRÁFICO 2 — MÉDIAS POR DANO
# ============================================================

def plot_metricas_medias_por_dano(
    df_long,
    metodos=("Park", "Autoencoder"),
    salvar=True
):
    aplicar_estilo_artigo()

    df_use = df_long[df_long["Metodo"].isin(metodos)].copy()

    danos = sorted(df_use["falha"].unique())

    fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), dpi=300)

    metricas = ["RMSD", "CCDM"]

    x = np.arange(len(danos))
    bar_w = 0.32

    offsets = np.linspace(
        -bar_w / 2,
        bar_w / 2,
        len(metodos)
    )

    for ax, metrica in zip(axes, metricas):

        for i, metodo in enumerate(metodos):
            vals = []
            stds = []

            for d in danos:
                mask = (
                    (df_use["Metodo"] == metodo) &
                    (df_use["falha"] == d)
                )

                vals.append(df_use.loc[mask, metrica].mean())
                stds.append(df_use.loc[mask, metrica].std())

            ax.bar(
                x + offsets[i],
                vals,
                width=bar_w,
                yerr=stds,
                capsize=4,
                edgecolor="black",
                linewidth=0.7,
                label=metodo
            )

        ax.set_xlabel("Classe de dano")
        ax.set_ylabel(metrica)

        ax.set_xticks(x)
        ax.set_xticklabels([f"Dano {d}" for d in danos])

        ax.grid(False)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_title("(a) RMSD médio por dano")
    axes[1].set_title("(b) CCDM médio por dano")

    axes[0].legend(
        frameon=True,
        facecolor="white",
        edgecolor="none"
    )

    plt.tight_layout()

    if salvar:
        nome = "Park_vs_AE_Metricas_medias_por_dano.png"
        caminho = os.path.join(OUTPUT_DIR, nome)

        plt.savefig(
            caminho,
            bbox_inches="tight",
            facecolor="white"
        )

        print(f"Figura salva em: {caminho}")

    plt.show()


# ============================================================
# 14) GRÁFICO 3 — CURVA EXEMPLO
# ============================================================

def plot_curva_exemplo(
    df_base,
    df_park,
    df_ae,
    y_ref,
    fcols,
    fhz,
    idx_show=None,
    falha=None,
    temperatura=None,
    salvar=True
):
    aplicar_estilo_artigo()

    if idx_show is None:
        df_aux = df_base.copy()

        if falha is not None:
            df_aux = df_aux[df_aux["falha"] == falha]

        if temperatura is not None:
            df_aux = df_aux[
                np.isclose(df_aux["temperatura_c"], temperatura)
            ]

        if len(df_aux) == 0:
            raise ValueError(
                "Nenhuma curva encontrada com os filtros escolhidos."
            )

        idx_show = df_aux.index[0]

    fhz_khz = fhz / 1e3

    T = df_base.loc[idx_show, "temperatura_c"]
    D = df_base.loc[idx_show, "falha"]

    plt.figure(figsize=(12, 6), dpi=300)

    plt.plot(
        fhz_khz,
        y_ref,
        "--",
        c="black",
        linewidth=1.2,
        label=f"Referência {REF_TEMP} °C"
    )

    plt.plot(
        fhz_khz,
        df_base.loc[idx_show, fcols],
        linewidth=1.0,
        alpha=0.70,
        label=f"Original — {T} °C — Dano {D}"
    )

    plt.plot(
        fhz_khz,
        df_park.loc[idx_show, fcols],
        linewidth=1.6,
        label="Park"
    )

    plt.plot(
        fhz_khz,
        df_ae.loc[idx_show, fcols],
        linewidth=1.8,
        label="Autoencoder"
    )

    plt.xlabel("Frequência (kHz)")
    plt.ylabel("Parte real da impedância")

    plt.title(
        f"Park vs Autoencoder — {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz"
    )

    plt.legend(
        frameon=True,
        facecolor="white",
        edgecolor="none"
    )

    plt.grid(False)

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    if salvar:
        nome = f"Park_vs_AE_Curva_idx{idx_show}_Dano{D}_Temp{T}.png"
        caminho = os.path.join(OUTPUT_DIR, nome)

        plt.savefig(
            caminho,
            bbox_inches="tight",
            facecolor="white"
        )

        print(f"Figura salva em: {caminho}")

    plt.show()

    print(f"Índice usado: {idx_show}")
    print(f"Temperatura: {T} °C")
    print(f"Dano: {D}")


# ============================================================
# 15) GRÁFICO 4 — BOXPLOT POR MÉTODO E DANO
# ============================================================

def plot_boxplot_metricas_por_dano(
    df_long,
    metodos=("Park", "Autoencoder"),
    salvar=True
):
    aplicar_estilo_artigo()

    df_use = df_long[df_long["Metodo"].isin(metodos)].copy()

    danos = sorted(df_use["falha"].unique())

    metricas = ["RMSD", "CCDM"]

    for metrica in metricas:
        plt.figure(figsize=(12, 6), dpi=300)

        labels = []
        data = []

        for d in danos:
            for metodo in metodos:
                vals = df_use.loc[
                    (df_use["falha"] == d) &
                    (df_use["Metodo"] == metodo),
                    metrica
                ].dropna().values

                data.append(vals)
                labels.append(f"D{d}\n{metodo}")

        plt.boxplot(
            data,
            labels=labels,
            showmeans=True
        )

        plt.ylabel(metrica)
        plt.title(f"{metrica} por dano — Park vs Autoencoder")

        plt.grid(False)

        ax = plt.gca()
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        plt.tight_layout()

        if salvar:
            nome = f"Park_vs_AE_Boxplot_{metrica}.png"
            caminho = os.path.join(OUTPUT_DIR, nome)

            plt.savefig(
                caminho,
                bbox_inches="tight",
                facecolor="white"
            )

            print(f"Figura salva em: {caminho}")

        plt.show()


# ============================================================
# 16) GRÁFICO 5 — LOSS DO AUTOENCODER
# ============================================================

def plot_ae_loss(history, salvar=True):
    aplicar_estilo_artigo()

    plt.figure(figsize=(10, 5), dpi=300)

    plt.plot(
        history["epoch"],
        history["train_loss"],
        linewidth=2,
        label="Treino"
    )

    plt.plot(
        history["epoch"],
        history["val_loss"],
        linewidth=2,
        label="Validação"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Treinamento do Autoencoder")

    plt.legend(
        frameon=True,
        facecolor="white",
        edgecolor="none"
    )

    plt.grid(False)

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    if salvar:
        nome = "Autoencoder_loss.png"
        caminho = os.path.join(OUTPUT_DIR, nome)

        plt.savefig(
            caminho,
            bbox_inches="tight",
            facecolor="white"
        )

        print(f"Figura salva em: {caminho}")

    plt.show()


# ============================================================
# 17) EXECUÇÃO GERAL
# ============================================================

def executar_comparacao_park_autoencoder():
    timings = {}

    print("====================================================")
    print("COMPARAÇÃO PARK vs AUTOENCODER")
    print("====================================================")

    # --------------------------------------------------------
    # Carregamento
    # --------------------------------------------------------
    t0 = time.time()

    df = pd.read_pickle(ARQ_BASE)

    fcols, fhz = get_freq_columns(
        df,
        FREQ_MIN_KHZ,
        FREQ_MAX_KHZ
    )

    y_ref = get_reference_curve(
        df,
        fcols,
        REF_TEMP
    )

    timings["load_reference"] = time.time() - t0

    print(f"\nTotal de amostras: {len(df)}")
    print(f"Amostras sem falha: {len(df[df['falha'] == 0])}")
    print(f"Faixa usada: {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
    print(f"Número de pontos de frequência: {len(fcols)}")
    print(f"Temperatura de referência: {REF_TEMP} °C")

    # --------------------------------------------------------
    # Métricas do sinal original
    # --------------------------------------------------------
    t0 = time.time()

    df_original_metricas = calcular_metricas(
        df_curvas=df,
        fcols=fcols,
        y_ref=y_ref,
        metodo="Original"
    )

    timings["original_metrics"] = time.time() - t0

    # --------------------------------------------------------
    # Park
    # --------------------------------------------------------
    t0 = time.time()

    df_park_curvas = compensar_park(
        df=df,
        fcols=fcols,
        fHz=fhz,
        y_ref=y_ref
    )

    df_park_metricas = calcular_metricas(
        df_curvas=df_park_curvas,
        fcols=fcols,
        y_ref=y_ref,
        metodo="Park"
    )

    timings["park"] = time.time() - t0

    # --------------------------------------------------------
    # Autoencoder
    # --------------------------------------------------------
    t0 = time.time()

    df_ae_curvas, ae_model, ae_history = compensar_autoencoder(
        df=df,
        fcols=fcols,
        fHz=fhz,
        y_ref=y_ref
    )

    df_ae_metricas = calcular_metricas(
        df_curvas=df_ae_curvas,
        fcols=fcols,
        y_ref=y_ref,
        metodo="Autoencoder"
    )

    timings["autoencoder"] = time.time() - t0

    # --------------------------------------------------------
    # DataFrame longo
    # --------------------------------------------------------
    df_long = pd.concat(
        [
            df_original_metricas,
            df_park_metricas,
            df_ae_metricas
        ],
        axis=0,
        ignore_index=False
    )

    # --------------------------------------------------------
    # Tabelas
    # --------------------------------------------------------
    metodos_comp = ("Park", "Autoencoder")

    tabela_resumo = resumo_geral(
        df_long,
        metodos=metodos_comp
    )

    tabela_temp = resumo_por_temperatura(
        df_long,
        metodos=metodos_comp
    )

    df_mono, resumo_mono = checar_monotonicidade(
        df_long,
        metodos=metodos_comp
    )

    # --------------------------------------------------------
    # Salvar tabelas
    # --------------------------------------------------------
    df_long.to_csv(
        os.path.join(OUTPUT_DIR, "df_long_metricas_Park_AE.csv"),
        index=False
    )

    tabela_temp.to_csv(
        os.path.join(OUTPUT_DIR, "resumo_por_temperatura_Park_AE.csv"),
        index=False
    )

    df_mono.to_csv(
        os.path.join(OUTPUT_DIR, "monotonicidade_Park_AE.csv"),
        index=False
    )

    resumo_mono.to_csv(
        os.path.join(OUTPUT_DIR, "resumo_monotonicidade_Park_AE.csv"),
        index=False
    )

    ae_history.to_csv(
        os.path.join(OUTPUT_DIR, "historico_loss_autoencoder.csv"),
        index=False
    )

    # --------------------------------------------------------
    # Prints
    # --------------------------------------------------------
    print("\n==================== RESUMO GERAL ====================")
    print(tabela_resumo)

    print("\n==================== MONOTONICIDADE ====================")
    print(resumo_mono)

    print("\n==================== TEMPOS ====================")
    for k, v in timings.items():
        print(f"{k:20s}: {v:.3f} s")

    print("\n✅ Comparação concluída.")

    return {
        "df_base": df,
        "df_park_curvas": df_park_curvas,
        "df_ae_curvas": df_ae_curvas,
        "df_original_metricas": df_original_metricas,
        "df_park_metricas": df_park_metricas,
        "df_ae_metricas": df_ae_metricas,
        "df_long": df_long,
        "tabela_resumo": tabela_resumo,
        "tabela_temp": tabela_temp,
        "df_mono": df_mono,
        "resumo_mono": resumo_mono,
        "y_ref": y_ref,
        "fcols": fcols,
        "fhz": fhz,
        "ae_model": ae_model,
        "ae_history": ae_history,
        "timings": timings
    }


# ============================================================
# 18) RODAR TUDO
# ============================================================

resultados = executar_comparacao_park_autoencoder()

df_base = resultados["df_base"]
df_park_curvas = resultados["df_park_curvas"]
df_ae_curvas = resultados["df_ae_curvas"]

df_original_metricas = resultados["df_original_metricas"]
df_park_metricas = resultados["df_park_metricas"]
df_ae_metricas = resultados["df_ae_metricas"]
df_long = resultados["df_long"]

tabela_resumo = resultados["tabela_resumo"]
tabela_temp = resultados["tabela_temp"]
df_mono = resultados["df_mono"]
resumo_mono = resultados["resumo_mono"]

y_ref = resultados["y_ref"]
fcols = resultados["fcols"]
fhz = resultados["fhz"]

ae_history = resultados["ae_history"]
timings = resultados["timings"]


# ============================================================
# 19) GERAR GRÁFICOS PRINCIPAIS
# ============================================================

metodos_comp = ("Park", "Autoencoder")

# Dano 0
plot_metricas_por_temperatura(
    df_long=df_long,
    dano=0,
    metodos=metodos_comp,
    salvar=True
)

# Dano 1
plot_metricas_por_temperatura(
    df_long=df_long,
    dano=1,
    metodos=metodos_comp,
    salvar=True
)

# Dano 2
plot_metricas_por_temperatura(
    df_long=df_long,
    dano=2,
    metodos=metodos_comp,
    salvar=True
)

# Médias por dano
plot_metricas_medias_por_dano(
    df_long=df_long,
    metodos=metodos_comp,
    salvar=True
)

# Boxplots
plot_boxplot_metricas_por_dano(
    df_long=df_long,
    metodos=metodos_comp,
    salvar=True
)

# Loss do autoencoder
plot_ae_loss(
    history=ae_history,
    salvar=True
)


# ============================================================
# 20) CURVAS EXEMPLO
# ============================================================

# Exemplo saudável
plot_curva_exemplo(
    df_base=df_base,
    df_park=df_park_curvas,
    df_ae=df_ae_curvas,
    y_ref=y_ref,
    fcols=fcols,
    fhz=fhz,
    falha=0,
    temperatura=48,
    salvar=True
)

# Exemplo dano 1
plot_curva_exemplo(
    df_base=df_base,
    df_park=df_park_curvas,
    df_ae=df_ae_curvas,
    y_ref=y_ref,
    fcols=fcols,
    fhz=fhz,
    falha=1,
    temperatura=55,
    salvar=True
)

# Exemplo dano 2
plot_curva_exemplo(
    df_base=df_base,
    df_park=df_park_curvas,
    df_ae=df_ae_curvas,
    y_ref=y_ref,
    fcols=fcols,
    fhz=fhz,
    falha=2,
    temperatura=55,
    salvar=True
)


# ============================================================
# 21) MOSTRAR TABELAS NO NOTEBOOK
# ============================================================

print("\n==================== TABELA RESUMO ====================")
display(tabela_resumo)

print("\n==================== RESUMO POR TEMPERATURA ====================")
display(tabela_temp)

print("\n==================== MONOTONICIDADE POR TEMPERATURA ====================")
display(df_mono)

print("\n==================== RESUMO DA MONOTONICIDADE ====================")
display(resumo_mono)

In [ ]:
# ============================================================
# HISTOGRAMAS + CURVAS EXEMPLO — PARK vs AUTOENCODER
# Colar depois do código principal Park vs Autoencoder
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# GARANTIA DE PASTA DE SAÍDA
# ============================================================

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = "Figuras_Park_vs_AE"
    os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# ESTILO DOS GRÁFICOS
# Se a função já existir no seu código anterior, pode deixar mesmo assim.
# ============================================================

def aplicar_estilo_artigo():
    plt.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": 18,
        "axes.labelsize": 20,
        "axes.titlesize": 20,
        "xtick.labelsize": 17,
        "ytick.labelsize": 17,
        "legend.fontsize": 15,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "pdf.fonttype": 42,
        "ps.fonttype": 42
    })


# ============================================================
# HISTOGRAMAS — PARK vs AUTOENCODER POR DANO
# ============================================================

def plot_histogramas_metricas_por_dano(
    df_long,
    dano=0,
    metodos=("Park", "Autoencoder"),
    bins=18,
    salvar=True
):
    aplicar_estilo_artigo()

    df_use = df_long[
        (df_long["falha"] == dano) &
        (df_long["Metodo"].isin(metodos))
    ].copy()

    metricas = ["RMSD", "CCDM"]

    fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), dpi=300)

    for ax, metrica in zip(axes, metricas):

        for metodo in metodos:
            vals = df_use.loc[
                df_use["Metodo"] == metodo,
                metrica
            ].dropna().values

            if len(vals) == 0:
                continue

            ax.hist(
                vals,
                bins=bins,
                alpha=0.55,
                edgecolor="black",
                linewidth=0.7,
                label=f"{metodo}"
            )

            ax.axvline(
                np.mean(vals),
                linestyle="--",
                linewidth=2,
                label=f"Média {metodo}"
            )

        ax.set_xlabel(metrica)
        ax.set_ylabel("Frequência")

        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_title(f"(a) Histograma RMSD — Dano {dano}")
    axes[1].set_title(f"(b) Histograma CCDM — Dano {dano}")

    axes[0].legend(
        frameon=True,
        facecolor="white",
        edgecolor="none"
    )

    plt.tight_layout()

    if salvar:
        nome = f"Park_vs_AE_Histogramas_Dano{dano}.png"
        caminho = os.path.join(OUTPUT_DIR, nome)

        plt.savefig(
            caminho,
            bbox_inches="tight",
            facecolor="white"
        )

        print(f"Figura salva em: {caminho}")

    plt.show()


# ============================================================
# HISTOGRAMAS EM GRADE — D0, D1 E D2 JUNTOS
# ============================================================

def plot_histogramas_metricas_grid(
    df_long,
    metodos=("Park", "Autoencoder"),
    bins=18,
    salvar=True
):
    aplicar_estilo_artigo()

    df_use = df_long[df_long["Metodo"].isin(metodos)].copy()

    danos = sorted(df_use["falha"].unique())
    metricas = ["RMSD", "CCDM"]

    fig, axes = plt.subplots(
        len(danos),
        2,
        figsize=(17, 5.2 * len(danos)),
        dpi=300
    )

    if len(danos) == 1:
        axes = np.array([axes])

    for i, dano in enumerate(danos):

        df_d = df_use[df_use["falha"] == dano]

        for j, metrica in enumerate(metricas):

            ax = axes[i, j]

            for metodo in metodos:
                vals = df_d.loc[
                    df_d["Metodo"] == metodo,
                    metrica
                ].dropna().values

                if len(vals) == 0:
                    continue

                ax.hist(
                    vals,
                    bins=bins,
                    alpha=0.55,
                    edgecolor="black",
                    linewidth=0.7,
                    label=metodo
                )

                ax.axvline(
                    np.mean(vals),
                    linestyle="--",
                    linewidth=2
                )

            ax.set_xlabel(metrica)
            ax.set_ylabel("Frequência")
            ax.set_title(f"{metrica} — Dano {dano}")

            ax.grid(False)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            if i == 0 and j == 0:
                ax.legend(
                    frameon=True,
                    facecolor="white",
                    edgecolor="none"
                )

    plt.tight_layout()

    if salvar:
        nome = "Park_vs_AE_Histogramas_Grid_Danos.png"
        caminho = os.path.join(OUTPUT_DIR, nome)

        plt.savefig(
            caminho,
            bbox_inches="tight",
            facecolor="white"
        )

        print(f"Figura salva em: {caminho}")

    plt.show()


# ============================================================
# SELEÇÃO AUTOMÁTICA DE ÍNDICE DE EXEMPLO
# ============================================================

def selecionar_indice_exemplo(df_base, falha, temperatura=None):
    df_aux = df_base[df_base["falha"] == falha].copy()

    if len(df_aux) == 0:
        raise ValueError(f"Nenhuma curva encontrada para falha = {falha}")

    if temperatura is None:
        return df_aux.index[0]

    df_temp = df_aux[np.isclose(df_aux["temperatura_c"], temperatura)]

    if len(df_temp) > 0:
        return df_temp.index[0]

    idx_mais_proximo = (
        (df_aux["temperatura_c"] - temperatura)
        .abs()
        .idxmin()
    )

    return idx_mais_proximo


# ============================================================
# CURVAS EXEMPLO EM GRADE — D0, D1 E D2
# ============================================================

def plot_curvas_exemplo_grid(
    df_base,
    df_park,
    df_ae,
    y_ref,
    fcols,
    fhz,
    exemplos=((0, 48), (1, 55), (2, 55)),
    salvar=True
):
    aplicar_estilo_artigo()

    fhz_khz = fhz / 1e3

    fig, axes = plt.subplots(
        len(exemplos),
        1,
        figsize=(13, 5.0 * len(exemplos)),
        dpi=300,
        sharex=True
    )

    if len(exemplos) == 1:
        axes = [axes]

    for ax, (falha, temperatura) in zip(axes, exemplos):

        idx_show = selecionar_indice_exemplo(
            df_base=df_base,
            falha=falha,
            temperatura=temperatura
        )

        T_real = df_base.loc[idx_show, "temperatura_c"]
        D_real = df_base.loc[idx_show, "falha"]

        ax.plot(
            fhz_khz,
            y_ref,
            "--",
            c="black",
            linewidth=1.1,
            label=f"Referência {REF_TEMP} °C"
        )

        ax.plot(
            fhz_khz,
            df_base.loc[idx_show, fcols],
            linewidth=1.0,
            alpha=0.65,
            label=f"Original — {T_real} °C"
        )

        ax.plot(
            fhz_khz,
            df_park.loc[idx_show, fcols],
            linewidth=1.7,
            label="Park"
        )

        ax.plot(
            fhz_khz,
            df_ae.loc[idx_show, fcols],
            linewidth=1.9,
            label="Autoencoder"
        )

        ax.set_ylabel("Parte real da impedância")

        ax.set_title(
            f"Curva exemplo — Dano {D_real} — {T_real} °C"
        )

        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.legend(
            frameon=True,
            facecolor="white",
            edgecolor="none"
        )

    axes[-1].set_xlabel("Frequência (kHz)")

    plt.tight_layout()

    if salvar:
        nome = "Park_vs_AE_Curvas_exemplo_grid.png"
        caminho = os.path.join(OUTPUT_DIR, nome)

        plt.savefig(
            caminho,
            bbox_inches="tight",
            facecolor="white"
        )

        print(f"Figura salva em: {caminho}")

    plt.show()


# ============================================================
# CURVA EXEMPLO INDIVIDUAL COM MÉTRICAS NO PRINT
# ============================================================

def plot_curva_exemplo_individual_com_metricas(
    df_base,
    df_park,
    df_ae,
    y_ref,
    fcols,
    fhz,
    falha=0,
    temperatura=48,
    salvar=True
):
    aplicar_estilo_artigo()

    idx_show = selecionar_indice_exemplo(
        df_base=df_base,
        falha=falha,
        temperatura=temperatura
    )

    fhz_khz = fhz / 1e3

    T_real = df_base.loc[idx_show, "temperatura_c"]
    D_real = df_base.loc[idx_show, "falha"]

    y_original = df_base.loc[idx_show, fcols].to_numpy(float)
    y_park = df_park.loc[idx_show, fcols].to_numpy(float)
    y_ae = df_ae.loc[idx_show, fcols].to_numpy(float)

    print("\n================ CURVA EXEMPLO ================")
    print(f"Índice usado: {idx_show}")
    print(f"Falha: {D_real}")
    print(f"Temperatura: {T_real} °C")

    print("\n--- Original ---")
    print(f"RMSD: {rmsd(y_original, y_ref):.6f}")
    print(f"CCDM: {ccdm(y_original, y_ref):.6f}")

    print("\n--- Park ---")
    print(f"RMSD: {rmsd(y_park, y_ref):.6f}")
    print(f"CCDM: {ccdm(y_park, y_ref):.6f}")

    print("\n--- Autoencoder ---")
    print(f"RMSD: {rmsd(y_ae, y_ref):.6f}")
    print(f"CCDM: {ccdm(y_ae, y_ref):.6f}")

    plt.figure(figsize=(12, 6), dpi=300)

    plt.plot(
        fhz_khz,
        y_ref,
        "--",
        c="black",
        linewidth=1.2,
        label=f"Referência {REF_TEMP} °C"
    )

    plt.plot(
        fhz_khz,
        y_original,
        linewidth=1.0,
        alpha=0.65,
        label=f"Original — {T_real} °C — Dano {D_real}"
    )

    plt.plot(
        fhz_khz,
        y_park,
        linewidth=1.7,
        label="Park"
    )

    plt.plot(
        fhz_khz,
        y_ae,
        linewidth=1.9,
        label="Autoencoder"
    )

    plt.xlabel("Frequência (kHz)")
    plt.ylabel("Parte real da impedância")

    plt.title(
        f"Exemplo de compensação — Dano {D_real} — {T_real} °C"
    )

    plt.legend(
        frameon=True,
        facecolor="white",
        edgecolor="none"
    )

    plt.grid(False)

    ax = plt.gca()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    if salvar:
        nome = f"Park_vs_AE_Curva_individual_Dano{D_real}_Temp{T_real}_idx{idx_show}.png"
        caminho = os.path.join(OUTPUT_DIR, nome)

        plt.savefig(
            caminho,
            bbox_inches="tight",
            facecolor="white"
        )

        print(f"Figura salva em: {caminho}")

    plt.show()


# ============================================================
# RODAR TODOS OS HISTOGRAMAS
# ============================================================

print("\n====================================================")
print("GERANDO HISTOGRAMAS PARK vs AUTOENCODER")
print("====================================================")

plot_histogramas_metricas_por_dano(
    df_long=df_long,
    dano=0,
    metodos=("Park", "Autoencoder"),
    bins=18,
    salvar=True
)

plot_histogramas_metricas_por_dano(
    df_long=df_long,
    dano=1,
    metodos=("Park", "Autoencoder"),
    bins=18,
    salvar=True
)

plot_histogramas_metricas_por_dano(
    df_long=df_long,
    dano=2,
    metodos=("Park", "Autoencoder"),
    bins=18,
    salvar=True
)

plot_histogramas_metricas_grid(
    df_long=df_long,
    metodos=("Park", "Autoencoder"),
    bins=18,
    salvar=True
)


# ============================================================
# RODAR CURVAS EXEMPLO EM GRADE
# ============================================================

print("\n====================================================")
print("GERANDO CURVAS EXEMPLO EM GRADE")
print("====================================================")

plot_curvas_exemplo_grid(
    df_base=df_base,
    df_park=df_park_curvas,
    df_ae=df_ae_curvas,
    y_ref=y_ref,
    fcols=fcols,
    fhz=fhz,
    exemplos=((0, 48), (1, 55), (2, 55)),
    salvar=True
)


# ============================================================
# RODAR CURVAS EXEMPLO INDIVIDUAIS
# ============================================================

print("\n====================================================")
print("GERANDO CURVAS EXEMPLO INDIVIDUAIS")
print("====================================================")

plot_curva_exemplo_individual_com_metricas(
    df_base=df_base,
    df_park=df_park_curvas,
    df_ae=df_ae_curvas,
    y_ref=y_ref,
    fcols=fcols,
    fhz=fhz,
    falha=0,
    temperatura=48,
    salvar=True
)

plot_curva_exemplo_individual_com_metricas(
    df_base=df_base,
    df_park=df_park_curvas,
    df_ae=df_ae_curvas,
    y_ref=y_ref,
    fcols=fcols,
    fhz=fhz,
    falha=1,
    temperatura=55,
    salvar=True
)

plot_curva_exemplo_individual_com_metricas(
    df_base=df_base,
    df_park=df_park_curvas,
    df_ae=df_ae_curvas,
    y_ref=y_ref,
    fcols=fcols,
    fhz=fhz,
    falha=2,
    temperatura=55,
    salvar=True
)

print("\n✅ Histogramas e curvas exemplo gerados com sucesso.")